### 1.Setup and Imports
First, we will import all the necessary libraries and set the device to use either GPU or CPU

In [1]:


import os
import json
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
from transformers import BertModel, BertTokenizer
import torchvision.models as models
from collections import defaultdict
import random
import numpy as np
import torch.nn.utils.rnn as rnn_utils


# Install and import sacrebleu, matplotlib, and scikit-learn\
try:
    import sacrebleu
except ImportError:
   print("Installing sacrebleu...")
   !pip install sacrebleu
   import sacrebleu

try:
    import matplotlib.pyplot as plt
except ImportError:
    print("Installing matplotlib...")
    !pip install matplotlib
    import matplotlib.pyplot as plt

try:
    from sklearn.metrics import accuracy_score, precision_score, f1_score, roc_curve, auc
except ImportError:
    print("Installing scikit-learn...")
    !pip install scikit-learn
    from sklearn.metrics import accuracy_score, precision_score, f1_score, roc_curve, auc

# Check for GPU and set the device\n
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


C:\Users\HP\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cpu


### 2. Model Architecture: Seq2Seq with Embedding Layer
Here, we'll define the improved Seq2SeqModel. The key change is adding an nn.Embedding layer in the decoder. "
This is crucial for reducing loss because it provides meaningful vector representations of the input tokens to the GRU, rather than simple one-hot vectors.


In [2]:

class Seq2SeqModel(nn.Module):
    """
    A sequence-to-sequence neural network for text generation from multimodal inputs.
    It combines a BERT encoder for text, a ResNet encoder for images, and a GRU decoder.
    """
    def __init__(self, num_classes, bert_model, cnn_model):
        super(Seq2SeqModel, self).__init__()
        # Encoders
        self.text_encoder = bert_model
        self.image_encoder = nn.Sequential(*list(cnn_model.children())[:-1])
        
        # Combined feature size from ResNet (2048) and BERT (768).
        combined_feature_size = 2048 + 768
        
        # Decoder: a GRU that takes the combined features as its initial hidden state.
        self.decoder = nn.GRU(
            input_size=self.text_encoder.config.hidden_size,
            hidden_size=combined_feature_size,
            batch_first=True
        )
        
        # Output layer maps the decoder's hidden state to the vocabulary size.
        self.out = nn.Linear(combined_feature_size, num_classes)
        
        # Embedding layer to convert token IDs to a vector representation.
        self.embeddings = nn.Embedding(num_classes, self.text_encoder.config.hidden_size)

    def forward(self, text_input, image_input, text_attention_mask, labels=None, teacher_forcing_ratio=0.5):
        # Get features from the encoders.
        text_features = self.text_encoder(text_input, attention_mask=text_attention_mask).last_hidden_state[:, 0, :]
        image_features = self.image_encoder(image_input).view(image_input.size(0), -1)
        
        # Concatenate features and prepare for the decoder.
        combined_features = torch.cat((text_features, image_features), dim=1).unsqueeze(0)
        
        # The decoding loop. During training, we use teacher forcing.
        if labels is not None and random.random() < teacher_forcing_ratio:
            decoder_input = self.embeddings(labels)
            output, hidden = self.decoder(decoder_input, combined_features)
        else:
            # Simplified inference path. A real implementation would use an iterative loop.
            decoder_input = torch.zeros_like(labels).unsqueeze(-1)
            output, hidden = self.decoder(decoder_input, combined_features)

        return self.out(output)

def pad_sequence_custom(batch):
    """
    A corrected custom collation function that properly pads sequences.
    
    This function takes a batch of data (text, image, labels) and pads the
    text and labels to a uniform length so they can be processed as a batch.
    It also generates the required attention mask for the text encoder.
    """
    text_inputs = [item[0] for item in batch]
    image_inputs = [item[1] for item in batch]
    labels = [item[2] for item in batch]

    # Pad the text sequences to a uniform length.
    padded_text_inputs = rnn_utils.pad_sequence(text_inputs, batch_first=True, padding_value=0)
    
    # Generate the attention mask. `1` for real tokens, `0` for padded tokens.
    text_attention_masks = (padded_text_inputs != 0).long()
    
    # Pad the labels. We use -100 as the padding value for CrossEntropyLoss.
    padded_labels = rnn_utils.pad_sequence(labels, batch_first=True, padding_value=-100)
    
    # Stack the image tensors. This works because images are all the same size.
    stacked_image_inputs = torch.stack(image_inputs)

    return padded_text_inputs, stacked_image_inputs, padded_labels, text_attention_masks


### 3. Dataset and Data Loading
This section defines the custom dataset class and a collate function to handle batches of variable-length sequences. No major changes were needed here.

In [3]:

class Seq2SeqDataset(torch.utils.data.Dataset):
    """
    A placeholder for your custom dataset class.
    
    This mock implementation simulates a dataset that returns text, image, and
    label tensors for a sequence-to-sequence task. In a real-world scenario,
    this class would load your actual data from disk.
    """
    def __init__(self, data_dir, tokenizer, labels_dir=None):
        self.tokenizer = tokenizer
        self.data_dir = data_dir
        self.labels_dir = labels_dir
        # Mock data to demonstrate functionality. In a real-world scenario,
        # you would load your data here.
        self.mock_data = [
            ("This is a short sentence.", "A short label."),
            ("This is a much longer sentence that needs to be padded correctly.", "A longer label for this sentence."),
            ("Hello world.", "Simple."),
        ]

    def __len__(self):
        return len(self.mock_data)

    def __getitem__(self, idx):
        # Your data loading logic here.
        text, label = self.mock_data[idx]
        
        # We return token IDs as lists so the collate function can pad them.
        text_tokens = self.tokenizer.encode(text, add_special_tokens=True)
        label_tokens = self.tokenizer.encode(label, add_special_tokens=True)
        
        # Mock image data
        image_data = torch.randn(3, 224, 224)
        
        return torch.tensor(text_tokens), image_data, torch.tensor(label_tokens)



### 4. Training and Evaluation Functions
These functions handle the core training and validation loops. The loss is calculated on the output of the decoder for each token, which is the correct way to train a sequence-to-sequence model.



In [4]:
def train_model(model, train_loader, val_loader, criterion, optimizer, num_epochs, device):
    """
    Trains and validates the Seq2Seq model.
    """
    model.train()
    train_losses = []
    val_losses = []
    
    for epoch in range(num_epochs):
        running_loss = 0.0
        
        # Training loop
        for i, (text_inputs, image_inputs, labels, attention_mask) in enumerate(train_loader):
            text_inputs = text_inputs.to(device)
            image_inputs = image_inputs.to(device)
            labels = labels.to(device)
            
            optimizer.zero_grad()
            
            outputs = model(text_inputs, image_inputs, attention_mask, labels)
            loss = criterion(outputs.view(-1, outputs.size(-1)), labels.view(-1))
            
            loss.backward()
            optimizer.step()
            
            if device.type == 'cuda':
                torch.cuda.empty_cache()
            
            running_loss += loss.item()
            
            if (i+1) % 10 == 0:
                print(f"Epoch [{epoch+1}/{num_epochs}], Batch [{i+1}/{len(train_loader)}], Loss: {loss.item():.4f}")
        
        epoch_loss = running_loss / len(train_loader)
        train_losses.append(epoch_loss)
        print(f"Epoch {epoch+1} Training Loss: {epoch_loss:.4f}")
        
        # Validation loop
        model.eval()
        val_loss = 0.0
        
        with torch.no_grad():
            for text_inputs, image_inputs, labels, attention_mask in val_loader:
                text_inputs = text_inputs.to(device)
                image_inputs = image_inputs.to(device)
                labels = labels.to(device)
                
                outputs = model(text_inputs, image_inputs, attention_mask, labels)
                loss = criterion(outputs.view(-1, outputs.size(-1)), labels.view(-1))
                val_loss += loss.item()
        
        val_epoch_loss = val_loss / len(val_loader)
        val_losses.append(val_epoch_loss)
        print(f"Epoch {epoch+1} Validation Loss: {val_epoch_loss:.4f}")
        
        model.train()
        
    return train_losses, val_losses




### 5. Main Execution and Hyperparameter Settings\n",
"This is the main block where you can initialize the model and start the training process. I've also added some comments to help you with hyperparameter tuning. You'll likely see a significant improvement in loss with the model changes, but experimenting with these values is key to finding the best performance. "



In [ ]:


# Define constants for the new directory structure.
DATASET_ROOT = "Split_Datasets"
TRAIN_DATA_DIR = os.path.join(DATASET_ROOT, "train")
VAL_DATA_DIR = os.path.join(DATASET_ROOT, "validation")
TEST_DATA_DIR = os.path.join(DATASET_ROOT, "test")
TEST_LABELS_DIR = os.path.join(DATASET_ROOT, "test_labels")

### Hyperparameters to Tune ###
NUM_EPOCHS = 10 
BATCH_SIZE = 8  # Reduced batch size to lower memory usage
LEARNING_RATE = 0.0005 # A lower learning rate can help with convergence
TEACHER_FORCING_RATIO = 0.5 # A key hyperparameter for Seq2Seq models

# Initialize tokenizer
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
NUM_CLASSES = tokenizer.vocab_size
print(f"Vocabulary size: {NUM_CLASSES}")

# Load data
train_dataset = Seq2SeqDataset(TRAIN_DATA_DIR, tokenizer)
val_dataset = Seq2SeqDataset(VAL_DATA_DIR, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=pad_sequence_custom, pin_memory=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=pad_sequence_custom, pin_memory=True, num_workers=4)

# Load pretrained models
bert_model_pretrained = BertModel.from_pretrained("bert-base-uncased").to(device)
cnn_model_pretrained = models.resnet50(weights=models.ResNet50_Weights.DEFAULT).to(device)

# Initialize the model, criterion, and optimizer

model = Seq2SeqModel(NUM_CLASSES, bert_model_pretrained, cnn_model_pretrained).to(device)
criterion = nn.CrossEntropyLoss(ignore_index=tokenizer.pad_token_id)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

# Start training
train_losses, val_losses = train_model(model, train_loader, val_loader, criterion, optimizer, NUM_EPOCHS, device)

# Load test data and labels from separate directories.
test_dataset = Seq2SeqDataset(TEST_DATA_DIR, tokenizer, labels_dir=TEST_LABELS_DIR)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=pad_sequence_custom)

# After training, you can evaluate on the test set
print("\nStarting evaluation on test set...")
model.eval()
test_total_loss = 0.0
with torch.no_grad():
    for text_inputs, image_inputs, labels in test_loader:
        text_inputs = text_inputs.to(device)
        image_inputs = image_inputs.to(device)
        labels = labels.to(device)
        
        outputs = model(text_inputs, image_inputs, labels)
        # Ensure outputs and labels have compatible shapes
        # Ensure outputs and labels have compatible shapes for loss calculation
        if outputs.dim() == 3 and labels.dim() == 2:
            # Reshape outputs and labels to match the expected dimensions
            loss = criterion(outputs.view(-1, outputs.size(-1)), labels.contiguous().view(-1))
        else:
            print(f"Unexpected dimensions: outputs.dim()={outputs.dim()}, labels.dim()={labels.dim()}")
            raise ValueError("Mismatch in dimensions between outputs and labels.")
        test_total_loss += loss.item()

test_avg_loss = test_total_loss / len(test_loader)
print(f"Test Loss: {test_avg_loss:.4f}")
print("\nEvaluation complete.")

# Code to save the trained model
model_save_path = "seq2seq_model.pth"
try:
    torch.save(model.state_dict(), model_save_path)
    print(f"\nModel saved successfully at {model_save_path}")
except Exception as e:
    print(f"\nError saving model: {e}")




Vocabulary size: 30522


C:\Users\HP\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


In [1]:
# Run the main training and evaluation code
# (All setup, training, and evaluation code is already present in previous cells.)
# Just execute the notebook cells in order to run your code.

### 6. Visualizations and Metrics
This section provides visual and quantitative metrics to evaluate the model's performance beyond just loss.



In [ ]:

def plot_losses(train_losses, val_losses):
    """
    Plots the training and validation losses over epochs.\n",
    """
    plt.figure(figsize=(10, 6))
    plt.plot(train_losses, label='Training Loss')
    plt.plot(val_losses, label='Validation Loss')
    plt.title('Training and Validation Loss Over Epochs')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)
    plt.show()

def evaluate_classification_metrics(model, loader, device):
    """
    Calculates token-level classification metrics for the model.\n",
    """
    model.eval()
    all_preds = []
    all_labels = []
    all_probs = []

    with torch.no_grad():
        for text_inputs, image_inputs, labels in loader:
            text_inputs = text_inputs.to(device)
            image_inputs = image_inputs.to(device)
            labels = labels.to(device)

            # Get outputs in training mode to get a probability distribution over the vocabulary\n",
            outputs = model(text_inputs, image_inputs, labels)
            
            # Flatten tensors to treat all tokens as a single classification task\n",
            outputs_flat = outputs.view(-1, outputs.size(-1))
            labels_flat = labels.view(-1)

            # Ignore padding tokens
            mask = labels_flat != 0 # Assuming 0 is pad token ID
            
            preds = torch.argmax(outputs_flat, dim=1)
            
            all_preds.extend(preds[mask].cpu().numpy())
            all_labels.extend(labels_flat[mask].cpu().numpy())
            
            # For ROC curve, get softmax probabilities
            probs = torch.softmax(outputs_flat, dim=1)
            all_probs.append(probs[mask].cpu().numpy())
            
    all_probs = np.concatenate(all_probs)

    # Calculate metrics. Note: 'average' is set to 'macro' to handle multi-class labels.\n",
    accuracy = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, average='macro', zero_division=0)
    f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)

    # Calculate ROC curve and AUC for each class (one-vs-rest)\n",
    n_classes = outputs.size(-1)
    fpr = defaultdict(list)
    tpr = defaultdict(list)
    roc_auc = defaultdict(list)
    
    y_test_bin = np.zeros((len(all_labels), n_classes), dtype=int)
    y_test_bin[np.arange(len(all_labels)), all_labels] = 1

    for i in range(n_classes):
        if np.any(y_test_bin[:, i]) and np.any(all_probs[:, i]):
            fpr[i], tpr[i], _ = roc_curve(y_test_bin[:, i], all_probs[:, i])
            roc_auc[i] = auc(fpr[i], tpr[i])

    return accuracy, precision, f1, fpr, tpr, roc_auc

def plot_roc_curve(fpr, tpr, roc_auc, n_classes):
    """
    Plots the multi-class ROC curves.
    """
    plt.figure(figsize=(10, 8))
    for i in range(n_classes):
        if i in fpr:
            plt.plot(fpr[i], tpr[i], label=f'ROC curve of class {i} (area = {roc_auc[i]:.2f})')
    plt.plot([0, 1], [0, 1], 'k--')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('Multi-class ROC Curve')
    plt.legend(loc="lower right")
    plt.show()

# Plotting the losses\n",
plot_losses(train_losses, val_losses)

# Calculating and displaying metrics
accuracy, precision, f1, fpr, tpr, roc_auc = evaluate_classification_metrics(model, test_loader, device)
print("\n--- Classification Metrics ---")
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision (macro): {precision:.4f}")
print(f"F1-Score (macro): {f1:.4f}")
print("\nClassification metric calculation complete.")

# Plotting the ROC curve
plot_roc_curve(fpr, tpr, roc_auc, NUM_CLASSES)
print("\nROC curve plotted.")





### 7. Advanced Evaluation with BLEU and BERT Similarity Scores
These metrics provide a more nuanced understanding of the generated text's quality. BLEU measures n-gram overlap with the reference, while a BERT similarity score measures semantic similarity based on deep contextual embeddings.



In [ ]:
def calculate_bert_similarity(text1, text2, bert_model, tokenizer, device):
    """
    Calculates the cosine similarity between the BERT embeddings of two texts.
    """
    # Tokenize and get embeddings
    inputs1 = tokenizer(text1, return_tensors='pt', truncation=True, padding=True).to(device)
    inputs2 = tokenizer(text2, return_tensors='pt', truncation=True, padding=True).to(device)
    
    with torch.no_grad():
        # Get the CLS token embedding as the sentence embedding
        embeddings1 = bert_model(**inputs1).last_hidden_state[:, 0, :]
        embeddings2 = bert_model(**inputs2).last_hidden_state[:, 0, :]

    # Calculate cosine similarity
    cosine_sim = nn.functional.cosine_similarity(embeddings1, embeddings2, dim=1)
    return cosine_sim.item()

def generate_and_evaluate(model, loader, tokenizer, bert_model, device):
    """
    Generates text from the model and evaluates it against the labels using BLEU and BERT similarity.
    """
    model.eval()
    hypotheses = []  # The model's generated sentences
    references = []  # The ground truth sentences
    similarity_scores = []

    with torch.no_grad():
        for text_inputs, image_inputs, labels in loader:
            for i in range(text_inputs.size(0)):
                single_text_input = text_inputs[i].unsqueeze(0).to(device)
                single_image_input = image_inputs[i].unsqueeze(0).to(device)
                
                # Generate output sequences (inference mode)
                generated_tokens = model(single_text_input, single_image_input)

                # Decode the generated tokens
                generated_text = tokenizer.decode(generated_tokens, skip_special_tokens=True)
                hypotheses.append(generated_text)

                # Decode the reference labels
                reference_text = tokenizer.decode(labels[i], skip_special_tokens=True)
                references.append(reference_text)
                
                # Calculate BERT similarity score
                sim_score = calculate_bert_similarity(generated_text, reference_text, bert_model, tokenizer, device)
                similarity_scores.append(sim_score)

    # sacrebleu expects a list of references for each hypothesis
    references_for_sacrebleu = [[ref] for ref in references]
    bleu = sacrebleu.corpus_bleu(hypotheses, references_for_sacrebleu)
    
    avg_similarity = np.mean(similarity_scores)
    return bleu.score, avg_similarity

print("\nCalculating BLEU and BERT Similarity scores on the test set...")
bleu_score, bert_similarity_score = generate_and_evaluate(model, test_loader, tokenizer, bert_model_pretrained, device)
print(f"Corpus BLEU Score: {bleu_score:.2f}")
print(f"Average BERT Similarity Score: {bert_similarity_score:.4f}")
print("\nBLEU and BERT score calculation complete.")




SyntaxError: invalid syntax (2415509836.py, line 13)


### 8. Inference and Model Generation
Now that the model is trained, you can use it to generate a marking scheme for a new question. This is a very simple example to show how it works.



In [ ]:

# Example of inference
model.eval()
with torch.no_grad():
    # Get an example from your validation set (you would use new data here)
    example_text, example_image, _ = next(iter(val_loader))
    example_text = example_text[0].unsqueeze(0).to(device)
    example_image = example_image[0].unsqueeze(0).to(device)

    # Generate the output sequence
    generated_tokens = model(example_text, example_image)
    
    # Decode the tokens back to text
    generated_text = tokenizer.decode(generated_tokens, skip_special_tokens=True)
    print("\nGenerated Marking Scheme:")
    print(generated_text)



metadata: {kernelspec: 
    {
        display_name: "Python 3",
        language: "python",
        name: "python3"
    },
    language_info: 
    {
        codemirror_mode: 
        {
            name: "ipython",
            version: 3
        },
        file_extension: ".py",
        mimetype: "text/x-python",
        name: "python",
        nbconvert_exporter: "python",
        pygments_lexer: "ipython3"
    }
},

"nbformat": 4,
"nbformat_minor": 5